In [1]:
!pip install albumentations segmentation-models-pytorch -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 3.8 MB/s eta 0:00:00


In [2]:
import os
import cv2
import numpy as np
from tqdm import tqdm

print("=== HỆ THỐNG XỬ LÝ DỮ LIỆU TỔNG HỢP (PHIÊN BẢN KAGGLE CHUẨN) ===")

TARGET_SIZE = (512, 512)

# 1. Định nghĩa đường dẫn theo chuẩn Kaggle
# Input: Nơi chứa dữ liệu gốc (Chỉ đọc)
# Working: Nơi lưu kết quả (Có quyền ghi)
KAGGLE_INPUT = '/kaggle/input/'
KAGGLE_WORKING = '/kaggle/working/'

combined_img_dir = os.path.join(KAGGLE_WORKING, 'dataset/train_combined/images/')
combined_mask_dir = os.path.join(KAGGLE_WORKING, 'dataset/train_combined/masks/')

os.makedirs(combined_img_dir, exist_ok=True)
os.makedirs(combined_mask_dir, exist_ok=True)

# Từ điển quy đổi
dg_color_to_id = {(255, 255, 0): 0, (0, 255, 255): 1, (255, 0, 255): 2, (0, 255, 0): 3, (255, 0, 0): 4, (255, 255, 255): 5}
loveda_to_dg = {1: 5, 2: 0, 3: 0, 4: 4, 5: 5, 6: 3, 7: 1}

# ==========================================
# PHẦN 1: XỬ LÝ DEEPGLOBE
# ==========================================
print("\n[1/2] Đang quét radar tìm DeepGlobe...")
dg_img_paths = []
for root, dirs, files in os.walk(KAGGLE_INPUT):
    if 'deepglobe' in root.lower() and 'train' in root.lower() and 'sample' not in root.lower():
        for file in files:
            if file.endswith('_sat.jpg'):
                dg_img_paths.append(os.path.join(root, file))

if not dg_img_paths:
    print("⚠️ Không tìm thấy ảnh DeepGlobe. Hãy kiểm tra mục Input bên phải!")
else:
    print(f"-> Tìm thấy {len(dg_img_paths)} ảnh DeepGlobe. Đang xử lý...")
    for img_path in tqdm(dg_img_paths):
        img_name = os.path.basename(img_path)
        mask_path = img_path.replace('_sat.jpg', '_mask.png')
        
        img = cv2.imread(img_path)
        mask_bgr = cv2.imread(mask_path)
        if img is None or mask_bgr is None: continue
            
        img_res = cv2.resize(img, TARGET_SIZE, interpolation=cv2.INTER_AREA)
        mask_res = cv2.resize(mask_bgr, TARGET_SIZE, interpolation=cv2.INTER_NEAREST)
        
        gray_mask = np.zeros(TARGET_SIZE, dtype=np.uint8)
        for color, class_id in dg_color_to_id.items():
            gray_mask[np.all(mask_res == color, axis=-1)] = class_id
            
        cv2.imwrite(os.path.join(combined_img_dir, f"DG_{img_name}"), img_res)
        cv2.imwrite(os.path.join(combined_mask_dir, f"DG_{img_name.replace('_sat.jpg', '_mask.png')}"), gray_mask)

# ==========================================
# PHẦN 2: XỬ LÝ LOVEDA
# ==========================================
print("\n[2/2] Đang quét radar tìm LoveDA...")
loveda_paths = []
for root, dirs, files in os.walk(KAGGLE_INPUT):
    if 'loveda' in root.lower() and 'train' in root.lower() and 'images_png' in root.lower():
        for file in files:
            if file.endswith('.png'):
                loveda_paths.append(os.path.join(root, file))

if not loveda_paths:
    print("⚠️ Không tìm thấy ảnh LoveDA. Hãy kiểm tra mục Input bên phải!")
else:
    print(f"-> Tìm thấy {len(loveda_paths)} ảnh LoveDA. Đang xử lý...")
    for img_path in tqdm(loveda_paths):
        img_name = os.path.basename(img_path)
        mask_path = img_path.replace('images_png', 'masks_png')
        
        img = cv2.imread(img_path)
        mask_gray = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        if img is None or mask_gray is None: continue
            
        img_res = cv2.resize(img, TARGET_SIZE, interpolation=cv2.INTER_AREA)
        mask_res = cv2.resize(mask_gray, TARGET_SIZE, interpolation=cv2.INTER_NEAREST)
        
        new_mask = np.zeros_like(mask_res)
        for l_id, dg_id in loveda_to_dg.items():
            new_mask[mask_res == l_id] = dg_id
            
        cv2.imwrite(os.path.join(combined_img_dir, f"LDA_{img_name.replace('.png', '.jpg')}"), img_res)
        cv2.imwrite(os.path.join(combined_mask_dir, f"LDA_{img_name}"), new_mask)

print(f"\n✅ XỬ LÝ XONG! Tổng cộng có: {len(os.listdir(combined_img_dir))} ảnh trong kho lưu trữ.")

=== HỆ THỐNG XỬ LÝ DỮ LIỆU TỔNG HỢP (PHIÊN BẢN KAGGLE CHUẨN) ===

[1/2] Đang quét radar tìm DeepGlobe...
-> Tìm thấy 803 ảnh DeepGlobe. Đang xử lý...


100%|██████████| 803/803 [02:42<00:00,  4.94it/s]



[2/2] Đang quét radar tìm LoveDA...
-> Tìm thấy 2522 ảnh LoveDA. Đang xử lý...


100%|██████████| 2522/2522 [03:19<00:00, 12.63it/s]


✅ XỬ LÝ XONG! Tổng cộng có: 3325 ảnh trong kho lưu trữ.


In [3]:
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
import cv2
import numpy as np
import segmentation_models_pytorch as smp
from tqdm import tqdm
import gc

# 1. THIẾT LẬP THAM SỐ
DATA_DIR = '/kaggle/working/dataset/train_combined/'
IMG_DIR = os.path.join(DATA_DIR, 'images')
MASK_DIR = os.path.join(DATA_DIR, 'masks')
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

torch.cuda.empty_cache()
gc.collect()

BATCH_SIZE = 8
EPOCHS = 40

# 2. LỚP NẠP DỮ LIỆU TỐI ƯU (Giữ nguyên cấu trúc chuẩn của bạn)
class KaggleCombinedDataset(Dataset):
    def __init__(self, img_dir, mask_dir, transform=None):
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.transform = transform
        self.images = [f for f in os.listdir(img_dir) if f.endswith('.jpg')]

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_name = self.images[idx]
        img_path = os.path.join(self.img_dir, img_name)
        
        if 'DG_' in img_name:
            mask_name = img_name.replace('_sat.jpg', '_mask.png')
        else:
            mask_name = img_name.replace('.jpg', '.png')
            
        mask_path = os.path.join(self.mask_dir, mask_name)

        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

        if mask is None:
            alt_mask_name = img_name.replace('.jpg', '.png')
            mask = cv2.imread(os.path.join(self.mask_dir, alt_mask_name), cv2.IMREAD_GRAYSCALE)
            
        if mask is None:
            raise FileNotFoundError(f"❌ Không tìm thấy mask cho ảnh: {img_name} tại {mask_path}")

        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented['image']
            mask = augmented['mask']

        return image, mask.long()

# ---------------------------------------------------------
# [R&D NÂNG CẤP 1]: CHIẾN LƯỢC TĂNG CƯỜNG DỮ LIỆU ĐA TỶ LỆ
# ---------------------------------------------------------
train_transform = A.Compose([
    # Nâng cấp 1: Zoom in/out (0.7x đến 1.3x) bằng ShiftScaleRotate.
    # scale_limit = (-0.3, 0.3) tương đương scale từ 70% đến 130%.
    A.ShiftScaleRotate(
        shift_limit=0.1, 
        scale_limit=(-0.3, 0.3), 
        rotate_limit=15, 
        border_mode=cv2.BORDER_CONSTANT, 
        value=0, # Viền đen cho phần ảnh bị hụt khi zoom out
        mask_value=0, 
        p=0.8
    ),
    
    # Đảm bảo output luôn chính xác 512x512 sau khi dịch/zoom
    A.PadIfNeeded(min_height=512, min_width=512, border_mode=cv2.BORDER_CONSTANT, value=0, mask_value=0),
    A.RandomCrop(height=512, width=512),
    
    # Nâng cấp 2: Bẻ cong không gian.
    A.GridDistortion(num_steps=5, distort_limit=0.3, p=0.5),
    
    # Nâng cấp 3: Nhiễu màu sắc.
    A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1, p=0.5),
    
    # Kỹ thuật cơ bản
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.Transpose(p=0.5),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

dataset = KaggleCombinedDataset(IMG_DIR, MASK_DIR, transform=train_transform)
train_loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)

# ---------------------------------------------------------
# [R&D NÂNG CẤP 2]: HYBRID DILATED CONVOLUTION (HDC)
# ---------------------------------------------------------
def inject_hdc_to_resnet(unet_model):
    """
    Hàm tiêm HDC vào Encoder ResNet-50.
    Layer 4 của ResNet-50 có 3 khối Bottleneck. Ta sẽ đổi lớp Conv 3x3 ở giữa 
    của từng khối thành Dilated Conv với tỷ lệ rỗng lần lượt là [1, 2, 5].
    """
    hdc_rates = [1, 2, 5]
    
    # Duyệt qua 3 block của layer cuối cùng trong encoder
    for i, block in enumerate(unet_model.encoder.layer4):
        rate = hdc_rates[i % len(hdc_rates)]
        
        # block.conv2 chính là lớp tích chập 3x3
        # Đặt padding = rate để kích thước tensor đầu ra không bị lệch pha với Decoder
        block.conv2.dilation = (rate, rate)
        block.conv2.padding = (rate, rate)
        
    print("🚀 [R&D] Đã tiêm thành công kiến trúc HDC [1, 2, 5] vào ResNet-50 Backbone!")
    return unet_model

# 4. KHỞI TẠO MÔ HÌNH
model = smp.Unet(
    encoder_name="resnet50",
    encoder_weights="imagenet", 
    in_channels=3,
    classes=6 
)

# KÍCH HOẠT HÀM R&D HDC NGAY SAU KHI KHỞI TẠO
model = inject_hdc_to_resnet(model).to(DEVICE)

# Hàm Loss (Giữ nguyên để kiểm soát biến số)
criterion_dice = smp.losses.DiceLoss(mode='multiclass', from_logits=True)
criterion_ce = nn.CrossEntropyLoss()

def combo_loss(pred, target): 
    return criterion_dice(pred, target) + criterion_ce(pred, target)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

# 5. TIẾN HÀNH RÈN LUYỆN
print(f"🚀 KÍCH HOẠT SIÊU MÁY TÍNH KAGGLE - PHIÊN BẢN R&D")
print(f"📚 Tổng số ảnh huấn luyện: {len(dataset)}")

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}", leave=True)
    
    for images, masks in loop:
        images, masks = images.to(DEVICE), masks.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(images)
        loss = combo_loss(outputs, masks)
        
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        loop.set_postfix(loss=loss.item())

    scheduler.step()

# LƯU THÀNH QUẢ CUỐI CÙNG
torch.save(model.state_dict(), "unet_resnet50_V4_Advanced_HDC.pth")
print("\n🎉 CHÚC MỪNG! ĐÃ LUYỆN XONG SIÊU MÔ HÌNH V4 HDC.")

/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)
/tmp/ipykernel_23/2382374241.py:72: UserWarning: Argument(s) 'value, mask_value' are not valid for transform ShiftScaleRotate
  A.ShiftScaleRotate(
/tmp/ipykernel_23/2382374241.py:83: UserWarning: Argument(s) 'value, mask_value' are not valid for transform PadIfNeeded
  A.PadIfNeeded(min_height=512, min_width=512, border_mode=cv2.BORDER_CONSTANT, value=0, mask_value=0),


config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/102M [00:00<?, ?B/s]

🚀 [R&D] Đã tiêm thành công kiến trúc HDC [1, 2, 5] vào ResNet-50 Backbone!
🚀 KÍCH HOẠT SIÊU MÁY TÍNH KAGGLE - PHIÊN BẢN R&D
📚 Tổng số ảnh huấn luyện: 3325


Epoch 40/40: 100%|██████████| 416/416 [05:28<00:00,  1.27it/s, loss=0.476]



🎉 CHÚC MỪNG! ĐÃ LUYỆN XONG SIÊU MÔ HÌNH V4 HDC.
